# 特征选择：Filter 与 Wrapper 方法对比

**香港城市大学 MSc Electronic Commerce · IS6400 课程项目（个人完成） · Notebook 3/4**

在 Iris 与 Airbnb 数据集上实践四类特征选择方法：卡方检验（chi2）、ANOVA F 值（f_classif/f_regression）等 Filter 方法，以及序列前向选择（Sequential Feature Selector）、随机森林递归特征消除（RFE）等 Wrapper 方法，并对比不同方法的选择结果与适用场景。

Question 1

In [1]:
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif,chi2,f_regression
import pandas as pd

iris = pd.read_csv('../data/iris.txt', header=None)
iris.columns = ['sepal-L', 'sepal-W', 'petal-L', 'petal-W', 'class']
features = ['sepal-L', 'sepal-W', 'petal-L', 'petal-W']
target = ['class']
X = iris[features]
y = iris['class']
selection = SelectKBest(chi2, k=2)
X_new = selection.fit_transform(X, y)
selected_features = X.columns[selection.get_support()]
print(selected_features)

Index(['petal-L', 'petal-W'], dtype='str')


f_classif represents the ANOVA F-value between each feature and the class label. It is used for classification.

It is a filter method.

Question 2

In [2]:
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
selection = SequentialFeatureSelector(model, n_features_to_select=2, direction='forward')
selection.fit(X, y)

selected_features = X.columns[selection.get_support()]
print(selected_features)

Index(['petal-L', 'petal-W'], dtype='str')


SequentialFeatureSelector is a wrapper features selection method. It selects features based on the perfprmance of a machine learning model. It starts with no features. It adds one feature at a time and evaluates the model performance. At each step, the feature that gives the best model perfomance is selected. The process continue until the required number of the features is selected.

Two optimal features are petal-L and petal-W.

Question 3

In [3]:
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_regression
import pandas as pd

Airbnb = pd.read_csv('../data/Airbnb.csv')

y = Airbnb['log_price']
X = Airbnb.drop(columns=['log_price', 'id'])
X = pd.get_dummies(X, drop_first=True)
X = X.fillna(X.median(numeric_only=True))
selection = SelectKBest(f_regression, k=8)
X_new = selection.fit_transform(X, y)
selected_features = X.columns[selection.get_support()]
print(selected_features)

Index(['accommodates', 'bathrooms', 'cleaning_fee', 'bedrooms', 'beds',
       'bed_type_Real Bed', 'cancellation_policy_strict', 'city_SF'],
      dtype='str')


chi2 is designed for classification tasks and works with non-negative features. The target here is a continuous variable, so it is not been used.

f-classif is designed for classidication tasks, where the target is a class label. And log-price is a continuous variable, f-regression shoule be used instead.

Question 4

In [4]:
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor( n_estimators=30, n_jobs=-1, max_features='sqrt')

selection = RFE( estimator=model, n_features_to_select=8, step=0.2)
selection.fit(X, y)
selected_features = X.columns[selection.support_]
print(selected_features)

Index(['accommodates', 'bathrooms', 'number_of_reviews',
       'review_scores_rating', 'bedrooms', 'beds', 'property_type_House',
       'city_SF'],
      dtype='str')


The results of Question3 and Question4 are similar. Both of them contain accommodates, bathrooms, bedrooms, beds and city-SF. However, Question3 also selectes bed_type_Real Bed, cancellation_policy_strict and cleaning_fee, and Question4 selects number_of_reviews, eview_scores_rating and property_type_House.

The difference mainly because the two methods use different feature selection mechanisms. SelectKBest with f-regression computes each feature seperately based on its stastistical relationshipwith log-price. Contrastly, RFE used a Rondom Forest model and repeatly removes less important features based on feature importance. Therefore, RFE can consider the features together in a model, so its selected features can be different from Question3.